###Load raw to struct

####Cleans and transforms reference/static Bronze tables into Silver layer - applies type casting, deduplication, and MERGE upserts keyed on reference IDs

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window

In [0]:
dbutils.widgets.text("pipeline_run_id","")

In [0]:
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")

####Customers

In [0]:
# Selecting the valid states from reference geolocation table
valid_states = spark.sql(f''' select distinct geolocation_state from ecom_ref_raw.geolocation  ''')
valid_states_list = [row[0] for row in valid_states.collect()]

# Reading the raw customers table
raw_customers = spark.sql(f''' select * from ecom_raw.customers ''')

# Replace any special characters with space and convert to lower case
customers = (raw_customers.withColumn("customer_city", 
                                     lower(trim(regexp_replace(col("customer_city"), "[^a-zA-Z\\s]", "")))
                                    )
                          .withColumn("customer_state",
                                      upper(trim(regexp_replace(col("customer_state"), "[^a-zA-Z]", "")))
                                    )
                          .withColumn("customer_zip_code_prefix",
                                      coalesce(col("customer_zip_code_prefix"), lit(00000))
                                    )
            )

# Replacing any other state with Unknown
customers = customers.withColumn("customer_state", 
                                 when(col("customer_state").isin(valid_states_list),col("customer_state"))
                                 .otherwise("Unknown")
                                )

# Removing nulls from Primary keys
customers = customers.filter( col("customer_id").isNotNull() & col("customer_unique_id").isNotNull())

# Removing any duplicates
customers = customers.withColumn("rnk", row_number().over(Window.partitionBy("customer_id").orderBy(col("ingestion_date").desc())))
customers = customers.filter(col("rnk") == 1)
customers = customers.drop("rnk")

# Checking the validity of row
customers = customers.withColumn("is_valid",when(col("customer_id").isNull(), False)
                                 .when(col("customer_state") == "Unknown", False)
                                 .otherwise(True)
                                )

# Add tracking columns
customers = (customers.withColumn("ingestion_date", current_timestamp())
                      .withColumn("source_file_name", lit("raw_customers"))
                      .withColumn("pipeline_run_id", lit(pipeline_run_id))
           )

customers.createOrReplaceTempView("customers")

In [0]:
# Data quality check
total   = customers.count()
valid   = customers.filter(col("is_valid") == True).count()
invalid = customers.filter(col("is_valid") == False).count()
print(f"customers total: {total}")
print(f"customers valid: {valid}")
print(f"customers invalid: {invalid}")

In [0]:
spark.sql('''
MERGE INTO ecom_struct.customers AS target
USING customers AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
''').display()

####Orders

In [0]:
# Read raw table
raw_orders = spark.sql(f''' select * from ecom_raw.orders ''')

# Cleaning
orders = (raw_orders
            .withColumn("order_purchase_timestamp",col("order_purchase_timestamp").cast(TimestampType()))
            .withColumn("order_approved_at",col("order_approved_at").cast(TimestampType()))
            .withColumn("order_delivered_carrier_date",col("order_delivered_carrier_date").cast(TimestampType()))
            .withColumn("order_delivered_customer_date",col("order_delivered_customer_date").cast(TimestampType()))
            .withColumn("order_estimated_delivery_date",col("order_estimated_delivery_date").cast(TimestampType()))
            .withColumn("order_year",year(col("order_purchase_timestamp")).cast(IntegerType()))
            .withColumn("order_month",month(col("order_purchase_timestamp")).cast(IntegerType()))
            .withColumn("order_status",when(col("order_status").isNotNull(),col("order_status")).otherwise("unknown"))
        )

# Remove null PKs and FKs
orders = orders.filter( 
                col("order_id").isNotNull() &
                col("customer_id").isNotNull() &
                col("order_purchase_timestamp").isNotNull()
        )

# Remove duplicates
orders = orders.withColumn("rnk", row_number().over(Window.partitionBy("order_id").orderBy(col("ingestion_date").desc())))
orders = orders.filter(col("rnk") == 1)
orders = orders.drop("rnk")

# Data quality flag
orders = orders.withColumn("is_valid",when(col("order_id").isNull(), False)
                    .when(col("customer_id").isNull(), False)
                    .when(col("order_purchase_timestamp").isNull(), False)
                    .otherwise(True)
        )

# Add tracking columns
orders = (orders.withColumn("ingestion_date", current_timestamp())
                .withColumn("source_file_name", lit("raw_orders"))
                .withColumn("pipeline_run_id", lit(pipeline_run_id))
        )
orders.createOrReplaceTempView("orders")

In [0]:
# Data quality check
total   = orders.count()
valid   = orders.filter(col("is_valid") == True).count()
invalid = orders.filter(col("is_valid") == False).count()
print(f"orders total: {total}")
print(f"orders valid: {valid}")
print(f"orders invalid: {invalid}")

In [0]:
spark.sql('''
MERGE INTO ecom_struct.orders AS target
USING orders AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
''').display()

####Order Items

In [0]:
# Read raw table
raw_order_items = spark.sql(f''' select * from ecom_raw.order_items ''')

# Cleaning
order_items = (raw_order_items.withColumn("order_item_id",col("order_item_id").cast(IntegerType())) 
                              .withColumn("price",col("price").cast(DoubleType())) 
                              .withColumn("freight_value",col("freight_value").cast(DoubleType())) 
                              .withColumn("shipping_limit_date",col("shipping_limit_date").cast(TimestampType()))
                )

# Validate price and freight
order_items = (order_items.filter(col("price") > 0).filter(col("freight_value") >= 0))

# Fill null freight with 0
order_items = order_items.fillna({"freight_value": 0.0})

# Remove null PKs
order_items = order_items.filter(
    col("order_id").isNotNull() &
    col("product_id").isNotNull() &
    col("price").isNotNull()
)

# Remove duplicates
order_items = order_items.withColumn("rnk", row_number().over(Window.partitionBy("order_id","order_item_id").orderBy(col("ingestion_date").desc())))
order_items = order_items.filter(col("rnk") == 1)
order_items = order_items.drop("rnk")

# Data quality flag
order_items = order_items.withColumn("is_valid",when(col("order_id").isNull(), False)
    .when(col("price") <= 0, False)
    .when(col("freight_value") < 0, False)
    .otherwise(True)
)

# Add tracking columns
order_items = (order_items.withColumn("ingestion_date", current_timestamp()) 
                          .withColumn("source_file_name", lit("raw_order_items")) 
                          .withColumn("pipeline_run_id", lit(pipeline_run_id))
            )
order_items.createOrReplaceTempView("order_items")

In [0]:
# Data quality check
total   = order_items.count()
valid   = order_items.filter(col("is_valid") == True).count()
invalid = order_items.filter(col("is_valid") == False).count()
print(f"order_items total: {total}")
print(f"order_items valid: {valid}")
print(f"order_items invalid: {invalid}")

In [0]:
spark.sql('''
MERGE INTO ecom_struct.order_items AS target
USING order_items AS source
ON target.order_id = source.order_id 
AND target.order_item_id = source.order_item_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
''').display()

####Order Payments

In [0]:
# Read raw table
raw_order_payments = spark.sql(f''' select * from ecom_raw.order_payments''')

# Cleaning
order_payments = (raw_order_payments.withColumn("payment_sequential",col("payment_sequential").cast(IntegerType()))
                           .withColumn("payment_installments",col("payment_installments").cast(IntegerType()))
                           .withColumn("payment_value",col("payment_value").cast(DoubleType()))
                           .withColumn("payment_type",when(col("payment_type").isNotNull(),col("payment_type")).otherwise("unknown"))
                )

# Validate payment value
order_payments = order_payments.filter(col("payment_value") > 0)

# Fill null installments with 1
order_payments = order_payments.fillna({"payment_installments": 1})

# Remove null PKs
order_payments = order_payments.filter(
    col("order_id").isNotNull() &
    col("payment_value").isNotNull()
)

# Remove duplicates
order_payments = order_payments.withColumn("rnk", row_number().over(Window.partitionBy("order_id","payment_sequential").orderBy(col("ingestion_date").desc())))
order_payments = order_payments.filter(col("rnk") == 1)
order_payments = order_payments.drop("rnk")

# Data quality flag
order_payments = order_payments.withColumn("is_valid",when(col("order_id").isNull(), False)
    .when(col("payment_value") <= 0, False)
    .when(col("payment_type") == "unknown", False)
    .otherwise(True)
)

# Add tracking columns
order_payments = (order_payments.withColumn("ingestion_date", current_timestamp())
                                .withColumn("source_file_name", lit("raw_order_payments"))
                                .withColumn("pipeline_run_id", lit(pipeline_run_id))
                )
order_payments.createOrReplaceTempView("order_payments")

In [0]:
# Data quality check
total   = order_payments.count()
valid   = order_payments.filter(col("is_valid") == True).count()
invalid = order_payments.filter(col("is_valid") == False).count()
print(f"int_order_payments total: {total}")
print(f"int_order_payments valid: {valid}")
print(f"int_order_payments invalid: {invalid}")

In [0]:
spark.sql('''
MERGE INTO ecom_struct.order_payments AS target
USING order_payments AS source
ON target.order_id = source.order_id 
AND target.payment_sequential = source.payment_sequential
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
''').display()

####Order Reviews

In [0]:
# Read raw table
raw_order_reviews = spark.sql(f''' select * from ecom_raw.order_reviews ''')

# Cleaning
order_reviews = raw_order_reviews.filter(
    (length(col("review_id")) == 32) & (col("review_id").rlike("^[a-zA-Z0-9]+$")) &
    (length(col("order_id")) == 32) & (col("order_id").rlike("^[a-zA-Z0-9]+$")) &
    (col("review_score").rlike("^[0-9]+$")) &
    (length(col("review_creation_date")) == 19) & (col("review_creation_date").rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")) &
    (length(col("review_answer_timestamp")) == 19) & (col("review_answer_timestamp").rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"))
)

order_reviews = (order_reviews.withColumn("review_score",col("review_score").cast(IntegerType()))
                              .withColumn("review_creation_date",col("review_creation_date").cast(TimestampType()))
                              .withColumn("review_answer_timestamp",col("review_answer_timestamp").cast(TimestampType()))
                              .withColumn("review_comment_title",coalesce(
                                      regexp_replace(col("review_comment_title"),"[^a-zA-Z0-9\\s]", ""),lit("No Title")))
                              .withColumn("review_comment_message",coalesce(
                                      regexp_replace(col("review_comment_message"),"[^a-zA-Z0-9\\s]", ""),lit("No Comment")))
                )

# Remove duplicates
order_reviews = order_reviews.withColumn("rnk", row_number().over(Window.partitionBy("review_id").orderBy(col("ingestion_date").desc())))
order_reviews = order_reviews.filter(col("rnk") == 1)
order_reviews = order_reviews.drop("rnk")


# Data quality flag
order_reviews = order_reviews.withColumn("is_valid",when(col("review_id").isNull(), False)
    .when(col("review_score").isNull(), False)
    .when(~col("review_score").between(1, 5), False)
    .otherwise(True)
)

# Add tracking columns
order_reviews = (order_reviews.withColumn("ingestion_date", current_timestamp())
                              .withColumn("source_file_name", lit("raw_order_reviews"))
                              .withColumn("pipeline_run_id", lit(pipeline_run_id))
                )
order_reviews.createOrReplaceTempView("order_reviews")

In [0]:
# Data quality check
total   = order_reviews.count()
valid   = order_reviews.filter(col("is_valid") == True).count()
invalid = order_reviews.filter(col("is_valid") == False).count()
print(f"order_reviews total: {total}")
print(f"order_reviews valid: {valid}")
print(f"order_reviews invalid: {invalid}")

In [0]:
spark.sql('''
MERGE INTO ecom_struct.order_reviews AS target
USING order_reviews AS source
ON target.review_id = source.review_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
''').display()

####Sellers

In [0]:
# Read raw table
raw_sellers = spark.sql(f''' select * from ecom_raw.sellers ''')

# Cleaning
sellers = (raw_sellers.withColumn("seller_city",lower(trim(regexp_replace(col("seller_city"), "[^a-zA-Z\\s]", ""))))
                      .withColumn("seller_state",upper(trim(regexp_replace(col("seller_state"), "[^a-zA-Z]", ""))))
                      .withColumn("seller_zip_code_prefix",coalesce(col("seller_zip_code_prefix"), lit("00000")))
        )

# Remove null PKs
sellers = sellers.filter(
    col("seller_id").isNotNull()
)

# Remove duplicates
sellers = sellers.withColumn("rnk", row_number().over(Window.partitionBy("seller_id").orderBy(col("ingestion_date").desc())))
sellers = sellers.filter(col("rnk") == 1)
sellers = sellers.drop("rnk")

# Data quality flag
sellers = sellers.withColumn("is_valid",
    when(col("seller_id").isNull(), False)
    .when(col("seller_state") == "Unknown", False)
    .otherwise(True)
)

# Add tracking columns
sellers = (sellers.withColumn("ingestion_date", current_timestamp())
                  .withColumn("source_file_name", lit("raw_sellers"))
                  .withColumn("pipeline_run_id", lit(pipeline_run_id))
        )

sellers.createOrReplaceTempView("sellers")

In [0]:
# Data quality check
total   = sellers.count()
valid   = sellers.filter(col("is_valid") == True).count()
invalid = sellers.filter(col("is_valid") == False).count()
print(f"int_sellers total: {total}")
print(f"int_sellers valid: {valid}")
print(f"int_sellers invalid: {invalid}")

In [0]:
# MERGE SQL
spark.sql(f'''
    MERGE INTO ecom_struct.sellers AS target
    USING sellers AS source
    ON target.seller_id = source.seller_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
''').display()